# 3. Dimensionality reduction and Clustering

This notebook documents the EDA, Dimensionality reduction and Clustering of the [OSMI Mental Health in Tech Survey 2016](https://www.kaggle.com/datasets/osmi/mental-health-in-tech-2016), according to _Task 1: Mental Health in Technology-related Jobs_ of the Unsupervised Learning and Feature Engineering course (DLBDSMLUSL01).

The written report is not part of the repository.  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

from dotenv import load_dotenv
from pathlib import Path
BASE_PATH = str(Path().cwd().parent.resolve() / "utils")
import os
import sys
sys.path.append(BASE_PATH)

from sklearn.manifold import MDS
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_samples, silhouette_score, adjusted_rand_score
from sklearn.preprocessing import OrdinalEncoder

from skbio.stats.distance import DistanceMatrix, mantel

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from scipy.spatial.distance import squareform, is_valid_dm

from statsmodels.stats.multitest import multipletests

from itertools import combinations

# This function is in this repository
from metrics import gower_matrix

In [ ]:
load_dotenv(Path().cwd().parent.joinpath(".env"))

In [ ]:
# Dataset location
LOCATION_DATSET = Path(os.getenv("LOCATION_DATASET")).parent

In [ ]:
df = pd.read_csv(LOCATION_DATSET.joinpath("osmi_mental_health_clean.csv"), index_col=0)

In [ ]:
#Verify no NaN values are in the dataset
df.isna().sum().sum()

In [ ]:
DATA_UTILS = Path(os.getenv("DATA_UTILS"))

## EDA

In [ ]:
n_samples, n_columns = df.shape

- 81% of surveyed worked wor for a company
- 19% are self-employed
- Age range is between 17 and 55 years old, with a mean of 33.6 years old. TODO: Test for statistical difference?
- Around 40% of surveyed worker currently have a mental health disorder.
- 50% of workers have been diagnosed before with a mental health disorder by a medical professional
- Almost 60% have seeked help -> High awareness in the population is likely
- Tech industry still has a majority of male workers
- Around 65% of female workers have been diagnosed with a mental health disorder in the past in contrast to 44% amongst the male workers. Among non.binary workers, 77% of them have been diagnosed in past.
- Female and non-binary workers have seeked help of a health professional before with 75% and 88% respectively, in contrast only 52% of men have done so

In [ ]:
df["What is your age?"].describe()

In [ ]:
# How many people are self-employed
fig, ax = plt.subplots(figsize=(6,6), nrows=1, ncols=1)
sns.countplot(data=df, x="Are you self-employed?", stat="percent", ax=ax)
ax.set_xticks([0,1])
ax.set_yticks(np.arange(0,110,10))
ax.set_ylim([0,100])
ax.set_xticklabels(["No", "Yes"], fontsize=14)
ax.set_xlabel(ax.get_xlabel(), fontsize=14)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)
ax.set_ylabel(ax.get_ylabel(), fontsize=14)
ax.grid(True, linewidth=0.5, linestyle=":", axis="y")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
fig.tight_layout()

In [ ]:
df[["What is your gender?", "What is your age?"]].groupby("What is your gender?").describe()

In [ ]:
fig, ax = plt.subplots(figsize=(6,6), nrows=1, ncols=1)
sns.histplot(df, x ="What is your age?", bins=15, kde=True, stat="density", alpha=0.5, ax=ax)

ax.grid(True, linewidth=0.5, linestyle=':', axis="y")
ax.spines[["top", "right", "bottom", "left"]].set_visible(False)
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)
ax.set_xlabel("Age (Years)", fontsize=14)
ax.set_ylabel(ax.get_ylabel(), fontsize=14)
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", pad=10)

In [ ]:
df_explore = df.copy()

In [ ]:
encoder = OrdinalEncoder(categories=[["No", "Yes", "Maybe"]])
df_explore["Do you currently have a mental health disorder?"] = encoder.fit_transform(df_explore[["Do you currently have a mental health disorder?"]]).astype(np.int64)

df_explore["Have you been diagnosed with a mental health condition by a medical professional?"] = encoder.transform(df_explore[["Have you been diagnosed with a mental health condition by a medical professional?"]].values).astype(np.int64)

In [ ]:
fig, axs = plt.subplots(figsize=(18,6), nrows=1, ncols=3)
sns.countplot(df_explore, x="Do you currently have a mental health disorder?", stat="percent", ax=axs[0])
axs[0].set_xticks([0,1,2])
axs[0].set_xticklabels(["No", "Yes", "Maybe"], fontsize=14)
sns.countplot(df_explore, x="Have you been diagnosed with a mental health condition by a medical professional?", stat="percent", ax=axs[1])
axs[1].set_xticks([0,1])
axs[1].set_xticklabels(["No", "Yes"], fontsize=14)
sns.countplot(df_explore, x="Have you ever sought treatment for a mental health issue from a mental health professional?", stat="percent", ax=axs[2])
axs[2].set_xticks([0,1])
axs[2].set_xticklabels(["No", "Yes"], fontsize=14)

for ax in axs:
    question = ax.get_xlabel().split()
    idx = len(question) // 2
    wrapped_question = f"{" ".join(question[:idx])}\n{" ".join(question[idx:])}"
    #ax.set_title(f"{ax.get_xlabel()}", wrap=True, fontsize=11)
    ax.grid(True, linewidth=0.5, linestyle=":", axis="y")
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.set_yticks(np.arange(0,70,10))
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)
    ax.tick_params(axis="y", length=0, pad=10)    
    ax.set_xlabel(wrapped_question, fontsize=14)
    ax.set_ylabel("(%)", fontsize=14)
fig.tight_layout()

In [ ]:
gender_count = df["What is your gender?"].value_counts()
gender_count

In [ ]:
current_disorders_by_gender = (
    df_explore
        .loc[:, ["What is your gender?", "Do you currently have a mental health disorder?"]]
        .groupby("What is your gender?")
        .value_counts(normalize=True)
)
current_disorders_by_gender = current_disorders_by_gender.to_frame().sort_index().reset_index()
current_disorders_by_gender

In [ ]:
diagnosed_by_gender = (
    df_explore
        .loc[:, ["What is your gender?", "Have you been diagnosed with a mental health condition by a medical professional?"]]
        .groupby("What is your gender?")
        .value_counts(normalize=True)
)
diagnosed_by_gender = diagnosed_by_gender.to_frame().sort_index().reset_index()
diagnosed_by_gender

In [ ]:
seeked_help_by_gender = (
    df_explore
        .loc[:, ["What is your gender?", "Have you ever sought treatment for a mental health issue from a mental health professional?"]]
        .groupby("What is your gender?")
        .value_counts(normalize=True)
)
seeked_help_by_gender = seeked_help_by_gender.to_frame().sort_index().reset_index()
seeked_help_by_gender

In [ ]:
question_1 = "Do you currently have a mental health disorder?"
question_2 = "Have you been diagnosed with a mental health condition by a medical professional?"
question_3 = "Have you ever sought treatment for a mental health issue from a mental health professional?"

fig, axs = plt.subplots(figsize=(18,6), nrows=1, ncols=3, sharey=True)

for question, feature, ax in zip((current_disorders_by_gender, diagnosed_by_gender, seeked_help_by_gender),(question_1, question_2, question_3), axs):

    sns.barplot(
        question,
        x=feature,
        y="proportion",
        hue="What is your gender?",
        palette="Dark2",
        ax=ax)

    caption = ax.get_xlabel().split()
    idx = len(caption) // 2
    wrapped_question = f"{" ".join(caption[:idx])}\n{" ".join(caption[idx:])}"
    
    ax.grid(True, linewidth=0.5, linestyle=":", axis="y")
    ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
    ax.tick_params(axis="y", length=0, pad=10)
    ax.set_xlabel(None)
    ax.set_xlabel(wrapped_question, fontsize=14)
    ax.set_yticks(np.arange(0,1.1,.1))
    ax.set_ylabel("(%)", fontsize=14)
    ax.legend(loc="upper left")
axs[0].set_xticks([0,1,2])
axs[0].set_xticklabels(["No", "Yes", "Maybe"], fontsize=14)
axs[1].set_xticks([0,1])
axs[1].set_xticklabels(["No", "Yes"], fontsize=14)
axs[2].set_xticks([0,1])
axs[2].set_xticklabels(["No", "Yes"], fontsize=14)
fig.tight_layout()

In [ ]:
roles = df["Which of the following best describes your work position?"].value_counts()
roles = (roles / roles.sum())*100

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sns.barplot(roles, orient="h", ax=ax)
ax.grid(True, linewidth=0.5, linestyle=":", axis="x")
ax.set_xticks(np.arange(0,45,5))
ax.set_xticklabels(ax.get_xticks(), fontsize=12)
ax.set_ylabel(None)
ax.set_xlabel("[%]", fontsize=14)
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="both", length=0, pad=10)

In [ ]:
encoder = OrdinalEncoder(categories=[["Never", "Sometimes", "Always"]])
df_explore["Do you work remotely?"] = encoder.fit_transform(df_explore[["Do you work remotely?"]]).astype(np.int64)

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
sns.countplot(df_explore, x="Do you work remotely?", stat="percent", ax=ax)

ax.grid(True, linewidth=0.5, linestyle=":", axis="y")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
ax.set_xlabel(None)
ax.set_ylabel("[%]")

In [ ]:
willing_toshare_mental_issue = df[df["Do you think that discussing a mental health disorder with your employer would have negative consequences?"]!= "Not applicable"]
n_samples, _ = willing_toshare_mental_issue.shape
print(n_samples)
willing_toshare_mental_issue["Do you think that discussing a mental health disorder with your employer would have negative consequences?"].value_counts() / n_samples

In [ ]:
willing_to_share_physical_issue = df[df["Do you think that discussing a physical health issue with your employer would have negative consequences?"]!= "Not applicable"]
n_samples, _ = willing_to_share_physical_issue.shape
print(n_samples)
willing_to_share_physical_issue["Do you think that discussing a physical health issue with your employer would have negative consequences?"].value_counts() / n_samples

## Feature selection

Pairwise $\chi^2$ tests and _Cramer's V_ statistic were used to identify highly associated features.  
As observed during the data cleaning, the missing values are not at random, they were driven by the survey design, e.g., questions conditional on _current_ or _previous_ employment.

The high values of Cramer's V in many cases, confirmed that the missingness carries meaningful information, for this reason, automatic feature selection based on the $\chi^2$ test results is avoided.  
Instead, features with high redundancy / strong association are reviewed and removed manually.

In [ ]:
categorical_columns = [col for col in df.columns if col != "What is your age?"]
results = []
for col1, col2 in combinations(categorical_columns, 2):
    table = pd.crosstab(df[col1], df[col2])
    chi2, pvalue, _, _ = chi2_contingency(table)
    v = association(table, method="cramer")
    result = {
        "feature1" : col1,
        "feature2" : col2,
        "chi2" : chi2,
        "pvalue" : pvalue,
        "cramers_v" : v
    }
    results.append(result)

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="cramers_v", ascending=False)
results_df

In [ ]:
redundant = results_df[results_df["cramers_v"] > 0.5][["feature1", "feature2", "cramers_v"]]
for i, s in redundant.iterrows():
    print(s["feature1"], "--", s["feature2"])
    print()

## Gower Distance

To be able to apply Dimensionality Reduction techniques to the data to get a better sense of it and be able to visualize it, the observations need to be transformed into distance metrics.  
The dataset includes mix data type: numeric, nominal and ordinal, to deal with this we use the Gower distance.

As a didactic exercise, the calculation of the Gower matrix has been implemented from scratch using numpy.  
Once the distance matrix has been calculated the data will be visualized using _Multidimensional Scaling_.

The idea of _Gower's distance_ is to measure how similar (or dissimilar) two observations are when the dataset contains _mixed types of variables_, such as numeric, ordinal, and categorical features.  
Gower's distance calculates a similarity score that is feature-specific and then combines the score for each feature into a single measure.

- __Numeric variables and ordinal variables__  
  The similarity between two observations is based on the absolute difference between the values of the observations for the respective feature, normalized by the range of the variable.

- __Nominal categorical variables__
  The similarity is __1__ if if both observations belong to the same category and __0__ if they belong to different categories.

The overall distance between the $i$ and $j$ observations is given by:

$$
\mathbf{D_{ij}} = 1 - \mathbf{S_{ij}}
$$

Where:  

- $\mathbf{S_{ij}}$ represents the overall similarity score between the observations $i$ and $j$ in the dataset.

The similarity score $\mathbf{S_{ij}}$ is calculated as:

$$
\mathbf{S_{ij}} = \frac{\sum_{k=1}^{p}( w_{ijk} \cdot s_{ijk})}{\sum_{k=1}^{p} w_{ijk}}
$$

Where:  

- $i$ and $j$ refer to the $i^{th}$ and $j^{th}$ observations in the dataset.
- $k$ refers to the index of the feature (the column in the dataset). 
- $s_{ijk}$ refers to the similarity score for the $i^{th}$ and $j^{th}$ observations, which is calculated depending on the data type of the feature $k$.  
- $w_{ijk}$ refers to the weight assigned to the similarity score. $w_{ijk} = 1$ for the features that can be compared and $w_{ijk} = 0$ for the ones that cannot be compared.

Missing values are handled automatically in this formulation, if one of both observations have a missing value for a feature $k$, they cannot be compared so the weight $w_{ijk} = 0$.

### Calculation of the similarity scores

1. For numerical and ordinal variables

$$
s_{ijk} = 1 - \frac{|x_{ik} - x_{jk}|}{max(x_k) - min(x_k)}
$$

2. For nominal variables

$$
s_{ijk} =
\begin{cases}
1, & \text{if } x_{ik} = x_{jk} \\
0, & \text{if } x_{ik} \neq x_{jk}
\end{cases}
$$

### Notes

In the implementation presented here, the edge case where one or more rows contain all NaN values, the returned matrix will show 1's for the corresponding row and column.

In [ ]:
to_drop = [
    'Are you self-employed?',
    'Do you have previous employers?',
    'Have you had a mental health disorder in the past?',
    'Would you have been willing to discuss a mental health issue with your previous co-workers?',
    'Would you feel comfortable discussing a mental health disorder with your coworkers?',
    'Did your previous employers provide resources to learn more about mental health issues and how to seek help?',
    'Does your employer provide mental health benefits as part of healthcare coverage?',
    'Does your employer offer resources to learn more about mental health concerns and options for seeking help?',
    'Were you aware of the options for mental health care provided by your previous employers?',
    'Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?'
]

In [ ]:
df_reduced = df.drop(to_drop, axis=1)
df_reduced.shape

In [ ]:
nominal_cols = [
    'Are you self-employed?',
    'How many employees does your company or organization have?',
    'Is your employer primarily a tech company/organization?',
    'Does your employer provide mental health benefits as part of healthcare coverage?',
    'Do you know the options for mental health care available under your employer-provided coverage?',
    'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?',
    'Does your employer offer resources to learn more about mental health concerns and options for seeking help?',
    'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?',
    'Do you think that discussing a mental health disorder with your employer would have negative consequences?',
    'Do you think that discussing a physical health issue with your employer would have negative consequences?',
    'Would you feel comfortable discussing a mental health disorder with your coworkers?',
    'Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?',
    'Do you feel that your employer takes mental health as seriously as physical health?',
    'Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?',
    'Do you have previous employers?',
    'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:',
    'Have your previous employers provided mental health benefits?',
    'Were you aware of the options for mental health care provided by your previous employers?',
    'Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?',
    'Did your previous employers provide resources to learn more about mental health issues and how to seek help?',
    'Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?',
    'Do you think that discussing a mental health disorder with previous employers would have negative consequences?',
    'Do you think that discussing a physical health issue with previous employers would have negative consequences?',
    'Would you have been willing to discuss a mental health issue with your previous co-workers?',
    'Would you have been willing to discuss a mental health issue with your direct supervisor(s)?',
    'Did you feel that your previous employers took mental health as seriously as physical health?',
    'Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?',
    'Would you be willing to bring up a physical health issue with a potential employer in an interview?',
    'Would you bring up a mental health issue with a potential employer in an interview?',
    'How willing would you be to share with friends and family that you have a mental illness?',
    'Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?',
    'Do you have a family history of mental illness?',
    'Have you had a mental health disorder in the past?',
    'Do you currently have a mental health disorder?',
    'Have you been diagnosed with a mental health condition by a medical professional?',
    'Have you ever sought treatment for a mental health issue from a mental health professional?',
    'If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?',
    'If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?',
    'Which of the following best describes your work position?',
    'What is your gender?'
]

ordinal_cols = {
    'Do you feel that being identified as a person with a mental health issue would hurt your career?' : ['No, it has not', "No, I don't think it would", 'Maybe', 'Yes, I think it would', 'Yes, it has'],
    'Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?' : ['No, they do not', "No, I don't think they would", 'Maybe','Yes, I think they would','Yes, they do'],
    'Do you work remotely?' : ['Never', 'Sometimes', 'Always']
}

numerical_cols = ['What is your age?']

In [ ]:
len(nominal_cols) + len(ordinal_cols) + len(numerical_cols)

As mentioned above, in the _Feature selection_ section, the missing answers in the dataset represent structure and are not Missing At Random.  
Two alternatives are explored here:

1. Calculation of the distance matrix and Non-Metric MDS using all features, i.e., 44 questions of the dataset.
2. Calculation of the distance matrix and Non-Metric MDS using a reduced number of features, in this case 34 features after manual feature selection.

### All questions included

In [ ]:
D = gower_matrix(
    df,
    nominal_cols=nominal_cols,
    ordinal_cols=ordinal_cols,
    numerical_cols=numerical_cols,
    as_frame=True
)

In [ ]:
is_valid_dm(D)

In [ ]:
mds = MDS(
    n_components=2,
    metric_mds=False,
    n_init=10,
    max_iter=500,
    metric="precomputed",
    init="random",
)

In [ ]:
D_transformed = mds.fit_transform(D)

In [ ]:
D_transformed

In [ ]:
# Store the transformed data
pd.DataFrame(D_transformed, columns=["feature1", "feature2"]).to_csv(DATA_UTILS.joinpath("NMDS_full.csv"), sep=";", index=False)

In [ ]:
# Load the NMDS using all features 
# Running NMDS takes long time so a copy of the reduced dataset has been stored
D_transformed = pd.read_csv(DATA_UTILS.joinpath("NMDS_full.csv"), sep=";").values

In [ ]:
D_transformed

### After manual feature selection

In [ ]:
D_reduced = gower_matrix(
    df_reduced,
    nominal_cols=nominal_cols,
    ordinal_cols=ordinal_cols,
    numerical_cols=numerical_cols,
    as_frame=True
)

In [ ]:
is_valid_dm(D_reduced)

In [ ]:
mds_reduced = MDS(
    n_components=2,
    metric_mds=False,
    n_init=10,
    max_iter=500,
    metric="precomputed",
    init="random"
)

In [ ]:
D_transformed_reduced = mds_reduced.fit_transform(D_reduced)

In [ ]:
D_transformed_reduced

In [ ]:
pd.DataFrame(D_transformed_reduced, columns=["feature1", "feature2"]).to_csv(DATA_UTILS.joinpath("NMDS_reduced.csv"), sep=";", index=False)

In [ ]:
# Load the NMDS using all features 
# Running NMDS takes long time so a copy of the reduced dataset has been stored
D_transformed_reduced = pd.read_csv(DATA_UTILS.joinpath("NMDS_reduced.csv"), sep=";").values

In [ ]:
D_transformed_reduced

## 4. Visualization

In [ ]:
_, n_features = df.shape
_, n_features_reduced = df_reduced.shape

with sns.axes_style("darkgrid"):
    fig, axs = plt.subplots(figsize=(16,6), nrows=1, ncols=2)

    sns.scatterplot(x=D_transformed[:,0],
                    y=D_transformed[:,1],
                    alpha=0.5,
                    ax=axs[0])
    axs[0].set_title(f"Non-metric MDS\nwith {n_features}")
    
    sns.scatterplot(x=D_transformed_reduced[:,0],
                    y=D_transformed_reduced[:,1],
                    alpha=0.5,
                    ax=axs[1])
    axs[1].set_title(f"Non-metric MDS\nwith {n_features_reduced}")
    
    for ax in axs:
        ax.set_xlabel(r"Feature 1")
        ax.set_ylabel(r"Feature 2")
        
fig.tight_layout()    

In [ ]:
def plot_dendrogram(model, **kwargs):
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_,
         model.distances_,
         counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)

### Mantel test

To assess whether the manual feature selection altered the structure of the data, a Mantel test is performed between _D_ and _D_reduced_.  

The _Mantel_ test compares two distance matrices (in this case _D_ and _D_reduced_) by calculating the correlation between the distances in the lower or open triangular portions of the matrices.
The test calculates the $r_M$ statistic given two distances matrices:

$$
r_{M} = \frac{1}{d-1} \sum_{i=1}^{n-1} \sum_{j=i+1}^{n} \text{stand}(D_X)_{ij} \text{ stand}(D_Y)_{ij}
$$

where:

$$
d = \frac{n(n-1)}{2}
$$

where:

- $n$ is the number of rows/columns of each of the distance matrices.
- $\text{stand}(D_X)_{ij} \text{ stand}(D_Y)_{ij}$ are the distance matrices with ther _upper triangules_ containing the standardized distances.

Mantel tests:

_Null hypothesis_ $H_0$ that there is no linear correlation between the two distance matrices. 
_Alternative hypothesis_ $H_a$: There distance matrices are lineraly correlated.

In [ ]:
gower = DistanceMatrix(D)

gower_reduced = DistanceMatrix(D_reduced)

r, pvalue, n = mantel(
    gower,
    gower_reduced,
    method="pearson"
)

In [ ]:
r

In [ ]:
pvalue

### Clusters

In [ ]:
model_complete = AgglomerativeClustering(
    n_clusters=3,
    metric="precomputed",
    linkage="complete",
    compute_full_tree=True,
    distance_threshold=None,
    compute_distances=True
).fit(D)

In [ ]:
fig, ax = plt.subplots(figsize=(60,18))
ax.grid(True, linewidth=1, linestyle=":", axis="y")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
ax.set_yticks(np.arange(0,1.1,.05))
plot_dendrogram(model_complete, ax=ax)

In [ ]:
model_complete_reduced = AgglomerativeClustering(
    n_clusters=3,
    metric="precomputed",
    linkage="complete",
    compute_full_tree=True,
    distance_threshold=None,
    compute_distances=True
).fit(D_reduced)

In [ ]:
fig, ax = plt.subplots(figsize=(60,18))
ax.grid(True, linewidth=1, linestyle=":", axis="y")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
ax.set_yticks(np.arange(0,1.1,.05))
plot_dendrogram(model_complete_reduced, ax=ax)

In [ ]:
model_average = AgglomerativeClustering(
    n_clusters=3,
    metric="precomputed",
    linkage="average",
    compute_full_tree=True,
    distance_threshold=None,
    compute_distances=True
).fit(D)

In [ ]:
fig, ax = plt.subplots(figsize=(60,18))
ax.grid(True, linewidth=1, linestyle=":", axis="y")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
ax.set_yticks(np.arange(0,1.1,.05))
plot_dendrogram(model_average, ax=ax)

In [ ]:
model_average_reduced = AgglomerativeClustering(
    n_clusters=3,
    metric="precomputed",
    linkage="average",
    compute_full_tree=True,
    distance_threshold=None,
    compute_distances=True
).fit(D_reduced)

In [ ]:
fig, ax = plt.subplots(figsize=(60,18))
ax.grid(True, linewidth=1, linestyle=":", axis="y")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
ax.set_yticks(np.arange(0,1.1,.05))
plot_dendrogram(model_average_reduced, ax=ax)

In [ ]:
model_single = AgglomerativeClustering(
    n_clusters=3,
    metric="precomputed",
    linkage="single",
    compute_full_tree=True,
    distance_threshold=None,
    compute_distances=True
).fit(D)

In [ ]:
fig, ax = plt.subplots(figsize=(60,18))
ax.grid(True, axis="y")
ax.set_yticks(np.arange(0,1.1,.05))
plot_dendrogram(model_single, ax=ax)

In [ ]:
model_single_reduced = AgglomerativeClustering(
    n_clusters=3,
    metric="precomputed",
    linkage="single",
    compute_full_tree=True,
    distance_threshold=None,
    compute_distances=True
).fit(D_reduced)

In [ ]:
ig, ax = plt.subplots(figsize=(60,18))
ax.grid(True, linewidth=1, linestyle=":", axis="y")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
ax.set_yticks(np.arange(0,1.1,.05))
plot_dendrogram(model_single_reduced, ax=ax)

### Clusters visualization

In [ ]:
models = {
    "complete" : model_complete,
    "average" : model_average,
    "single" : model_single,
    "complete_reduced" : model_complete_reduced,
    "average_reduced" : model_average_reduced,
    "single_reduced" : model_single_reduced
}

In [ ]:
with sns.axes_style("whitegrid"):
    fig, axs = plt.subplots(figsize=(21,12), nrows=2, ncols=3)
    for model, ax in zip(models.keys(), axs.flatten()):
        ax.set_title(f"Non-metric MDS\nLinkage: {model}")
        ax.set_xlabel(r"Feature 1")
        ax.set_ylabel(r"Feature 2")
        if "reduced" in model:
            sns.scatterplot(x=D_transformed_reduced[:,0],
                            y=D_transformed_reduced[:,1],
                            alpha=0.5, hue=models[model].labels_,
                            palette="Dark2",
                            ax=ax)
        else:
            sns.scatterplot(x=D_transformed[:,0],
                            y=D_transformed[:,1],
                            alpha=0.5,
                            hue=models[model].labels_,
                            palette="Dark2",
                            ax=ax)
    fig.tight_layout()

### Compare clusters formed with reduced features

In [ ]:
# ARI between
adjusted_rand_score(model_average.labels_, model_average_reduced.labels_)

In [ ]:
adjusted_rand_score(model_complete.labels_, model_complete_reduced.labels_)

In [ ]:
cluster_results = []
cluster_results_reduced = []
for k in [2,3,4,5,6,7,8]:
    for linkage_metric in ["complete", "average"]:
        linkage_full = AgglomerativeClustering(
            n_clusters=k,
            metric="precomputed",
            linkage=linkage_metric,
            compute_full_tree=True,
            distance_threshold=None,
            compute_distances=True
        ).fit(D)
        
        cluster_results.append(linkage_full)
        
        linkage_reduced = AgglomerativeClustering(
            n_clusters=k,
            metric="precomputed",
            linkage=linkage_metric,
            compute_full_tree=True,
            distance_threshold=None,
            compute_distances=True
        ).fit(D_reduced)

        cluster_results_reduced.append(linkage_reduced)

In [ ]:
for full, reduced in zip(cluster_results, cluster_results_reduced):
    print(f"Linkage:\nFull features: {full.linkage}; n_clusters: {full.n_clusters}\nReduced features: {reduced.linkage}; n_clusters {reduced.n_clusters}")
    print(f"ARI: {adjusted_rand_score(full.labels_, reduced.labels_)}")
    print()

In [ ]:
final_clusters = cluster_results[3]

In [ ]:
silhouette_scores_average = []
silhouette_scores_complete = []
for linkage in cluster_results:
    if linkage.linkage == "average":
        score = silhouette_score(D, linkage.labels_, metric="precomputed")
        silhouette_scores_average.append(score)
    elif linkage.linkage == "complete":
        score = silhouette_score(D, linkage.labels_, metric="precomputed")
        silhouette_scores_complete.append(score)

In [ ]:
fig, ax = plt.subplots()
ax.plot(
    np.arange(2,9,1),
    silhouette_scores_average,
    marker='.',
    label="Average Linkage"
)
ax.plot(
    np.arange(2,9,1),
    silhouette_scores_complete,
    marker='.',
    label="Complete Linkage"
)
ax.legend(loc="upper right")
ax.set_title("Silhouette scores for Average and Complete Linkages")
ax.grid(True, linestyle=":", linewidth=0.5)
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="both", length=0, pad=10)

### Final Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
sns.scatterplot(
    x=D_transformed[:,0],
    y=D_transformed[:,1],
    alpha=0.5,
    hue=final_clusters.labels_,
    palette="Dark2",
    ax=ax)

ax.set_title("NMDS visualization of the \nOSMI Mental Health in Tech Survey 2016")
ax.grid(True, linestyle=":", linewidth=0.5)
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="both", length=0, pad=10)

## Describing the clusters

In [ ]:
df_cluster = df.copy()

In [ ]:
df_cluster["cluster"] = final_clusters.labels_

In [ ]:
df_cluster["cluster"].value_counts(normalize=True)

In [ ]:
def cluster_profile(question, df):
    return (
        df.groupby("cluster")[question]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
        .pivot_table(values="proportion", index="cluster", columns=question)
    )

## Cluster characteristics

### General notes

- Clusters 0 and 2 consist of salaried employees, while cluster 1 is composed exclusively of self-employed workers.
- In both clusters 0 and 2, over 90% of respondents reported that they had not heard of or observed negative consequences for coworkers who disclosed mental health issues.
- Workers in clusters 0 and 1 reported similarly low awareness of mental health care options provided by their previous employers.
- Respondents in clusters 0 and 1 similarly reported that previous employers generally did not treat mental health with the same seriousness as physical health.
- Across all clusters, the majority of respondents stated that they would not disclse a mental health issue to a potential employer during a job interview.
- Family and friends represent the primary support system across clusters, with more than half of respondents reporting being somewhat to very open about sharing mental health issues with family or friends.
- Comparable proportions of workers in clusters 0 and 1 reported having been diagnosed with a mental health condition.
- No substantial differences in age distributions were observed across clusters; boxplots show considerable overlap.
- Gender distributions were comparable across all clusters.

### Cluster 0 – Mid-size tech employees with higher openness and lived experience

- Predominantly employed in small to medium-sized companies.
- Report greater availability of employer-provided mental health benefits compared to cluster 2.
- Approximately 26% reported that their employer offers resources to learn about mental health, while 28% were unaware of such resources, suggesting limited visibility rather than absence.
- Slightly higher confidence in employer-provided anonymity when accessing mental health or substance abuse resources.
- Greater confidence in the ease of obtaining medical leave for mental health reasons; only 9.9% reported that it would be very difficult.
- Lower perceived risk of negative consequences from discussing mental health with employers compared to cluster 2.
- Approximately two-thirds reported being comfortable or potentially comfortable discussing mental health issues with coworkers.
- A similar proportion reported being comfortable or potentially comfortable discussing mental health issues with supervisors.
- All respondents in this cluster reported having had previous employers.
- A higher proportion reported that previous employers formally discussed mental health as part of campaigns or official communications.
- More positive experiences with previous employers providing mental health learning resources compared to clusters 1 and 2.
- Despite higher exposure to negative experiences in past workplaces, respondents in this cluster reported comparatively greater trust in employers.
- A higher proportion reported personal or family history of mental illness, higher diagnosis rates, and higher treatment-seeking behavior.
- Predominantly composed of technically specialized roles, such as back-end and front-end developers.

### Cluster 1 – Self-employed workers with extensive prior exposure and limited institutional support

- The vast majority reported having previous employers.
- Fewer respondents reported that previous employers formally addressed mental health through campaigns or official communications.
- Reported limited access to employer-provided mental health learning resources in previous workplaces.
- Slightly higher trust in previous employers compared to cluster 0, despite fewer formal initiatives.
- A larger proportion reported having observed negative consequences for coworkers who disclosed mental health issues.
- Moderate concern that disclosure of mental health issues could negatively impact career prospects.
- Higher prevalence of unsupportive responses to mental health issues in past workplaces compared to other clusters.
- High prevalence of personal and family history of mental illness, diagnosis, and treatment-seeking.
- Predominantly composed of self-employed individuals fulfilling multiple roles (“one-person shops”).

### Cluster 2 – Early-career employees in large organizations with lower psychological safety

- Predominantly employed in large organizations.
- Report fewer employer-provided mental health benefits compared to cluster 0.
- Lower reported availability of mental health learning resources, though awareness levels were similar or slightly higher than in cluster 0.
- Lower confidence in employer protection of anonymity when accessing mental health or substance abuse resources.
- Lower confidence in the ease of obtaining medical leave for mental health reasons.
- Higher perceived risk of negative consequences from discussing mental health issues with employers compared to cluster 0.
- More reserved attitudes toward discussing mental health issues with coworkers and supervisors.
- Composed exclusively of workers without previous employers.
- Higher concern that being identified as a person with a mental health issue could negatively affect career prospects.
- Despite reporting fewer negative experiences in the workplace, respondents in this cluster expressed the lowest levels of trust in employers.
- Lower prevalence of personal and family history of mental illness, diagnosis, and treatment-seeking compared to clusters 0 and 1.

In [ ]:
for question in df_cluster.columns:
    if question == "cluster":
        continue
    elif question == "What is your age?":
        profile = df_cluster.groupby("cluster")[question].describe().reset_index()
    else:
        profile = cluster_profile(question, df_cluster)
    display(profile)   

In [ ]:
results_independence_clusters = []
for question in df_cluster.columns:
    if question == "cluster" or question == "What is your age?":
        continue
    contingency_table = pd.crosstab(df_cluster["cluster"], df_cluster[question])
    chi2, pvalue, _, _ = chi2_contingency(contingency_table)
    v = association(contingency_table, method="cramer")

    result_independence = {
        "question" : question,
        "p_value" : pvalue,
        "chi2" : chi2,
        "cramers_v" : v
    }
    results_independence_clusters.append(result_independence)

results_independence = pd.DataFrame(results_independence_clusters)
results_independence = results_independence.sort_values(by="p_value", ascending=True)

reject, pvalue_corrected, _, _ = multipletests(
    results_independence["p_value"].values,
    alpha=0.05,
    method="fdr_bh"
)

results_independence["reject_h0"] = reject
results_independence["p_value_corrected"] = pvalue_corrected

In [ ]:
results_independence

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sns.boxplot(
    df_cluster,
    x="What is your age?",
    hue="cluster",
    palette="Dark2",
    ax=ax
)
ax.grid(True, linestyle=":", linewidth=0.5, axis="x")
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="both", length=0, pad=10)